In [1]:
!pip install chromadb sentence-transformers psutil

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.0 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found 

In [2]:
import json
import re
import ast
from sentence_transformers import SentenceTransformer
import chromadb
import torch

# Configuración
JSONL_PATH = "time_series_generation.jsonl"
CHROMA_DB_PATH = "./memoria_maestro"
COLLECTION_NAME = "time_series_generation"



# Compilar regex una sola vez (más rápido)
PATTERN_LISTA = re.compile(
    r'\[\s*(?:-?\d+(?:\.\d+)?|\'[\dA-Za-z.-]+\'|"[\dA-Za-z.-]+")'
    r'(?:\s*,\s*(?:-?\d+(?:\.\d+)?|\'[\dA-Za-z.-]+\'|"[\dA-Za-z.-]+"))*\s*\]'
)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Modelo cargado usando: {device}")

# Modelo (cambia a uno más rápido si quieres)
model = SentenceTransformer('all-mpnet-base-v2', device=device)
# O el original: model = SentenceTransformer('all-mpnet-base-v2')

# Cliente Chroma
client = chromadb.PersistentClient(path=CHROMA_DB_PATH)
try:
    client.delete_collection(COLLECTION_NAME)
except:
    pass
collection = client.create_collection(name=COLLECTION_NAME)





Modelo cargado usando: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:

# --- Funciones auxiliares optimizadas ---

import torch
import psutil

def get_dynamic_embed_batch(device='cpu', default=32):
    """
    Devuelve un tamaño de lote recomendado para generar embeddings,
    basado en el dispositivo y la memoria disponible.

    Args:
        device (str): 'cuda', 'mps' o 'cpu'
        default (int): valor por defecto si no se puede determinar

    Returns:
        int: tamaño de lote sugerido
    """
    if device == 'cuda' and torch.cuda.is_available():
        try:
            free_mem, total_mem = torch.cuda.mem_get_info()
            free_gb = free_mem / (1024**3)
            # Heurística según VRAM libre (ajusta los umbrales a tu gusto)
            if free_gb > 10:
                return 64
            elif free_gb > 6:
                return 48
            elif free_gb > 4:
                return 32
            else:
                return 16
        except:
            return default

    elif device == 'mps':
        # En Apple Silicon (MPS) no hay una API directa para memoria GPU,
        # pero la memoria es unificada. Podemos usar la RAM total como referencia.
        try:
            ram_gb = psutil.virtual_memory().total / (1024**3)
            if ram_gb > 32:
                return 64
            elif ram_gb > 16:
                return 32
            else:
                return 16
        except:
            return default

    else:  # CPU
        try:
            ram_gb = psutil.virtual_memory().total / (1024**3)
            if ram_gb > 32:
                return 32   # En CPU, lotes muy grandes pueden saturar la RAM
            elif ram_gb > 16:
                return 16
            else:
                return 8
        except:
            return default

def get_dynamic_chroma_batch():
    ram_gb = psutil.virtual_memory().total / (1024**3)
    if ram_gb > 32: return 200
    elif ram_gb > 16: return 100
    else: return 50

def extract_all_time_series(text):
    """Extrae todas las listas usando ast.literal_eval (más rápido y robusto)."""
    series = []
    for match in PATTERN_LISTA.finditer(text):
        try:
            # literal_eval convierte directamente a lista de Python
            lst = ast.literal_eval(match.group())
            # Asegurar que es lista (por si acaso)
            if isinstance(lst, list):
                series.append(lst)
        except (SyntaxError, ValueError):
            # Fallback al método antiguo (casi nunca ocurrirá)
            pass
    return series

def clean_description(text):
    """Elimina las listas y espacios extra de forma más eficiente."""
    # Un solo re.sub con patrón compilado
    return re.sub(r'\s+', ' ', PATTERN_LISTA.sub('', text)).strip()

def build_search_text(entry):
    domain = entry['metadata']['original_domain']
    task = entry['metadata']['task_type']
    question = clean_description(entry['input'])
    response = entry['output']
    return f"Dominio: {domain}. Tarea: {task}. Pregunta: {question}. Respuesta: {response}."



ids_batch = []
embeddings_batch = []
metadatas_batch = []
documents_batch = []

def flush_batch():
    """Inserta el lote acumulado en Chroma y limpia las listas."""
    if ids_batch:
        collection.add(
            ids=ids_batch,
            embeddings=embeddings_batch,
            metadatas=metadatas_batch,
            documents=documents_batch
        )
        ids_batch.clear()
        embeddings_batch.clear()
        metadatas_batch.clear()
        documents_batch.clear()




In [4]:
EMBEDDING_BATCH_SIZE = get_dynamic_embed_batch(device)
BATCH_SIZE = get_dynamic_chroma_batch()
print(f"Tamaño de lote para Chroma: {BATCH_SIZE}")
print(f"Tamaño de lote para embeddings: {EMBEDDING_BATCH_SIZE}")


Tamaño de lote para Chroma: 50
Tamaño de lote para embeddings: 64


In [5]:
# --- Procesamiento por lotes ---
# Leer y procesar en lotes de embeddings
with open(JSONL_PATH, 'r', encoding='utf-8') as f:
    lines_buffer = []   # Para acumular textos de búsqueda y metadatos
    entries_buffer = [] # Para guardar los datos de cada entrada

    for line in f:
        entry = json.loads(line)

        # 1. Extraer series (con ast.literal_eval)
        series_input = extract_all_time_series(entry['input'])
        series_output = extract_all_time_series(entry['output'])
        series_data = {
            "input": [{"longitud_puntos": len(x), "serie": x} for x in series_input],
            "output": [{"longitud_puntos": len(x), "serie": x} for x in series_output]
        }
        series_json = json.dumps(series_data, separators=(',', ':'))

        # 2. Construir texto para embedding
        search_text = build_search_text(entry)

        # Guardar temporalmente para el lote de embeddings
        lines_buffer.append(search_text)
        entries_buffer.append({
            "id": entry['id'],
            "metadata": {
                "id": entry['id'],
                "domain": entry['metadata']['original_domain'],
                "task_type": entry['metadata']['task_type'],
                "question": entry['input'],
                "response": entry['output'],
                "time_series": series_json
                # search_text no se guarda en metadata (ya está en documents)
            }
        })
               # Cuando el buffer alcanza el tamaño de lote, procesamos los embeddings
        if len(lines_buffer) >= EMBEDDING_BATCH_SIZE:
            embeddings = model.encode(lines_buffer, batch_size=EMBEDDING_BATCH_SIZE, show_progress_bar=False)

            for i, emb in enumerate(embeddings):
                ids_batch.append(entries_buffer[i]["id"])
                embeddings_batch.append(emb.tolist())
                metadatas_batch.append(entries_buffer[i]["metadata"])
                documents_batch.append(lines_buffer[i])

            # Si el lote de Chroma también está lleno, lo insertamos
            if len(ids_batch) >= BATCH_SIZE:
                flush_batch()

            # Limpiar buffers temporales
            lines_buffer.clear()
            entries_buffer.clear()

    # Procesar el último lote (si queda algo)
    if lines_buffer:
        embeddings = model.encode(lines_buffer, batch_size=EMBEDDING_BATCH_SIZE, show_progress_bar=True)
        for i, emb in enumerate(embeddings):
            ids_batch.append(entries_buffer[i]["id"])
            embeddings_batch.append(emb.tolist())
            metadatas_batch.append(entries_buffer[i]["metadata"])
            documents_batch.append(lines_buffer[i])

        # Insertar resto
        flush_batch()

# Si quedaron registros en el buffer de Chroma (por si no se llegó a BATCH_SIZE)
flush_batch()

print(f"Índice construido con {collection.count()} documentos en {CHROMA_DB_PATH}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Índice construido con 192827 documentos en ./memoria_maestro


In [6]:
import shutil
import os

# Define the directory to be zipped and the output zip file name
directory_to_zip = CHROMA_DB_PATH
output_zip_file = f"{directory_to_zip}_batch.zip"

# Create the zip archive
shutil.make_archive(directory_to_zip, 'zip', directory_to_zip)

print(f"Successfully created {output_zip_file}")
print(f"You can download it from the file browser.")

Successfully created ./memoria_maestro_batch.zip
You can download it from the file browser.
